In [93]:
import numpy as np
import tensorflow as tf
import pandas as pd
from bkm10 import BKM10
from config import FIXED_CFFS
import time

In [96]:
fn = 'finalised pseudodata.csv'
df_all = pd.read_csv(fn)

names = ['ReH','ReHt','ReE','ReEt','ImH','ImHt','ImE','ImEt']
ranges = [[-30., 30.], 
            [-30., 30.], 
            [-100., 100.], 
            [-400., 400.], 
            [-20., 20.],
            [-30., 30.],
            [-100., 100.],
            [-400., 400.]]

In [57]:
def get_sig(plus, mins, phi, cffs):

    splus = plus.calculate_cross_section(phi, cffs)
    smins = mins.calculate_cross_section(phi, cffs)

    dsig = 0.5*(splus+smins)
    delsig = 0.5*(splus-smins)

    return dsig, delsig

def MSE(y1, y1_true, y2, y2_true):
    return tf.reduce_sum(tf.abs(y1-y1_true)**2+tf.abs(y2-y2_true)**2, axis=-1)

In [97]:
def run(set_num, restarts, epochs, replicas):
    
    df = df_all[df_all['set']==set_num].copy()

    plus = BKM10(*df[['k', 'Q2', 'xB', 't']].values[0], helc=1)
    mins = BKM10(*df[['k', 'Q2', 'xB', 't']].values[0], helc=-1)
    
    cffs = df[names].to_numpy()[0]

    phi = np.linspace(min(df['phi']), max(df['phi']), 20)
    phi = np.pi-np.deg2rad(phi)
    phi = tf.convert_to_tensor(phi, dtype=tf.float32)

    true_cffs = tf.convert_to_tensor(cffs, dtype=tf.float32)

    dsig_true, delsig_true = get_sig(plus, mins, phi, tf.expand_dims(true_cffs, axis=0))

    dsig_true = tf.tile(tf.expand_dims(dsig_true, axis=0), [replicas,1,1])
    delsig_true = tf.tile(tf.expand_dims(delsig_true, axis=0), [replicas,1,1])

    range_mins = tf.constant([ranges[i][0] for i in range(8)], dtype=tf.float32)
    range_maxs = tf.constant([ranges[i][1] for i in range(8)], dtype=tf.float32)

    predicted_cffs = tf.Variable(
        tf.random.uniform([replicas*restarts, 8], dtype=tf.float32) * (range_maxs - range_mins) + range_mins)

    optimizer = tf.keras.optimizers.Adam(learning_rate=1., epsilon=1e-12)

    @tf.function
    def train_step():
        with tf.GradientTape() as tape:
            dsig_pred, delsig_pred = get_sig(plus, mins, phi, predicted_cffs)
            dsig_pred = tf.reshape(dsig_pred, [replicas, restarts, -1])
            delsig_pred = tf.reshape(delsig_pred, [replicas, restarts, -1])
            losses = MSE(dsig_pred, dsig_true, delsig_pred, delsig_true)
        gradient = tape.gradient(losses, predicted_cffs)
        optimizer.apply_gradients([(gradient, predicted_cffs)])
        return losses

    start = time.perf_counter()
    for i in range(epochs):
        loss = train_step()
        # if (i % 100 == 0):
        #     print(i, loss.numpy(), free_val.numpy())
    # print(loss.numpy(), predicted_cffs.numpy())

    predicted_cffs = tf.reshape(predicted_cffs, [replicas, restarts, -1])
    end = time.perf_counter()
    print(f"Elapsed time: {end - start:.6f} seconds")

    return loss, predicted_cffs

loss, cffs = run(set_num= 4, restarts=10000, epochs= 1000, replicas=20)

Elapsed time: 23.834796 seconds


In [112]:
free_cff_indices = [i for i, name in enumerate(names) if name not in FIXED_CFFS]

best_indices = tf.argmin(loss, axis=1)
best_cffs = tf.gather(cffs, best_indices, batch_dims=1) 
best_loss = tf.gather(loss, best_indices, batch_dims=1) 
for bb,ll in zip(best_cffs, best_loss):
    print([float(bb[i]) for i in free_cff_indices], ll.numpy())

[-2.5149717330932617, 1.3477216958999634, 3.1965017318725586, 1.511767029762268] 2.4305905e-13
[-2.514859437942505, 1.3474656343460083, 3.20204496383667, 1.5011413097381592] 5.3037436e-14
[-2.514812469482422, 1.3473639488220215, 3.2039995193481445, 1.49734628200531] 1.15775445e-14
[-2.5147786140441895, 1.347289800643921, 3.2056713104248047, 1.4941259622573853] 6.110737e-14
[-2.514848232269287, 1.3474442958831787, 3.202162265777588, 1.5008717775344849] 6.813994e-15
[-2.5147244930267334, 1.347167730331421, 3.208627462387085, 1.4884374141693115] 2.2760613e-13
[-2.5148251056671143, 1.3473918437957764, 3.2033605575561523, 1.4985731840133667] 5.0272286e-15
[-2.5148253440856934, 1.3473882675170898, 3.2034687995910645, 1.4983690977096558] 5.2284566e-15
[-2.51495099067688, 1.3476734161376953, 3.1974542140960693, 1.5099354982376099] 1.726154e-13
[-2.5148370265960693, 1.3474187850952148, 3.202731132507324, 1.4997806549072266] 3.3410774e-15
[-2.514882802963257, 1.347524642944336, 3.200469255447387